In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
words = open('names.txt', 'r').read().splitlines()
words

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn',
 'abigail',
 'emily',
 'elizabeth',
 'mila',
 'ella',
 'avery',
 'sofia',
 'camila',
 'aria',
 'scarlett',
 'victoria',
 'madison',
 'luna',
 'grace',
 'chloe',
 'penelope',
 'layla',
 'riley',
 'zoey',
 'nora',
 'lily',
 'eleanor',
 'hannah',
 'lillian',
 'addison',
 'aubrey',
 'ellie',
 'stella',
 'natalie',
 'zoe',
 'leah',
 'hazel',
 'violet',
 'aurora',
 'savannah',
 'audrey',
 'brooklyn',
 'bella',
 'claire',
 'skylar',
 'lucy',
 'paisley',
 'everly',
 'anna',
 'caroline',
 'nova',
 'genesis',
 'emilia',
 'kennedy',
 'samantha',
 'maya',
 'willow',
 'kinsley',
 'naomi',
 'aaliyah',
 'elena',
 'sarah',
 'ariana',
 'allison',
 'gabriella',
 'alice',
 'madelyn',
 'cora',
 'ruby',
 'eva',
 'serenity',
 'autumn',
 'adeline',
 'hailey',
 'gianna',
 'valentina',
 'isla',
 'eliana',
 'quinn',
 'nevaeh',
 'ivy',
 'sadie',
 'piper',
 'lydia',
 'alexa',
 'josephine',
 'emery',
 'julia'

In [6]:
# vocab building

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)

In [7]:
block_size = 3 # 3 chars to predict 4th

def build_dataset(words):

    X, Y = [],[]
    for w in words:
        context = [0] * block_size
        # print(context)
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix] # 1: to crop and append only last 3 
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])
Xtr.shape

torch.Size([182580, 3])

In [19]:
# Summary code
n_emb = 10 # number embedding dimenstions
n_hidden = 200 #number of neurons in hidden layer

g = torch.Generator().manual_seed(2147483647)
C = torch.randn(vocab_size,n_emb, generator = g) #Embedding matrix
w1 = torch.randn(n_emb * block_size,n_hidden, generator = g) #Weights for first 200 neuron layer
b1 = torch.randn(n_hidden, generator = g) #bias for 200 newurons
w2 = torch.randn(n_hidden,vocab_size, generator = g) #Weights for first 200 neuron layer
b2 = torch.randn(vocab_size, generator = g) #bias for 200 newurons
parameters = [C, w1,b1,w2,b2]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

11897


In [20]:
# MODIFICATIONS TO IMPROVE THE NET
n_emb = 10 # number embedding dimenstions
n_hidden = 200 #number of neurons in hidden layer

g = torch.Generator().manual_seed(2147483647)
C = torch.randn(vocab_size,n_emb, generator = g) #Embedding matrix
# w1 = torch.randn(n_emb * block_size,n_hidden, generator = g) * 0.1 #Weights for first 200 neuron layer
w1 = torch.randn(n_emb * block_size,n_hidden, generator = g) * (5/3) / ((n_emb * block_size)**0.5) #Rather than multiply by 0.1, use kaiming method
b1 = torch.randn(n_hidden, generator = g) * 0.01 #bias for 200 newurons
w2 = torch.randn(n_hidden,vocab_size, generator = g) * 0.1 # multiply to scale down and get logits close to 0, so that everything has equal probability at first example
# Why cant we set weights to 0? -> 
b2 = torch.randn(vocab_size, generator = g) * 0 # multiply to scale down and get logits close to 0, so that everything has equal probability at first example
parameters = [C, w1,b1,w2,b2]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

11897


In [21]:
max_steps = 200000
batch_size = 32
lossi = []


for i in range(max_steps):
    # Forward pass
    ix = torch.randint(0, Xtr.shape[0], [batch_size,], generator = g)
    Xb, Yb = Xtr[ix], Ytr[ix]
    
    emb = C[Xb] # embedding
    embcat = emb.view(emb.shape[0],-1)
    h = torch.tanh(embcat @ w1 + b1) #first layer output
    logits = h @ w2 + b2 # second layer output
    loss = F.cross_entropy(logits, Ytr[ix]) # use this rather than do the negative log likelihood repeat

    
    # backward
    for p in parameters:
        p.grad = None
    loss.backward()
    lr = 0.1 if i<100000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    if i% 10000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():4f}')
    lossi.append(loss.log10().item())

      0/ 200000: 3.519319
  10000/ 200000: 2.515565
  20000/ 200000: 2.372989
  30000/ 200000: 2.121928
  40000/ 200000: 2.030331
  50000/ 200000: 2.333460
  60000/ 200000: 2.233984
  70000/ 200000: 2.097839
  80000/ 200000: 2.023001
  90000/ 200000: 2.066922
 100000/ 200000: 2.383280
 110000/ 200000: 2.151462
 120000/ 200000: 2.026450
 130000/ 200000: 2.317361
 140000/ 200000: 2.151475
 150000/ 200000: 2.316925
 160000/ 200000: 2.105258
 170000/ 200000: 2.016549
 180000/ 200000: 2.367981
 190000/ 200000: 2.066693


In [33]:
# MODIFICATIONS TO IMPROVE THE NET
n_emb = 10 # number embedding dimenstions
n_hidden = 200 #number of neurons in hidden layer

g = torch.Generator().manual_seed(2147483647)
C = torch.randn(vocab_size,n_emb, generator = g) #Embedding matrix
# w1 = torch.randn(n_emb * block_size,n_hidden, generator = g) * 0.1 #Weights for first 200 neuron layer
w1 = torch.randn(n_emb * block_size,n_hidden, generator = g) * (5/3) / ((n_emb * block_size)**0.5) #Rather than multiply by 0.1, use kaiming method
b1 = torch.randn(n_hidden, generator = g) * 0.01 #bias for 200 newurons
w2 = torch.randn(n_hidden,vocab_size, generator = g) * 0.1 # multiply to scale down and get logits close to 0, so that everything has equal probability at first example
# Why cant we set weights to 0? -> 
b2 = torch.randn(vocab_size, generator = g) * 0 # multiply to scale down and get logits close to 0, so that everything has equal probability at first example

bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))
bnmean_running = torch.zeros((1, n_hidden)) # because initially well have normalised set

parameters = [C, w1,b1,w2,b2,bngain,bnbias]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

12297


In [34]:
# batch norm

max_steps = 200000
batch_size = 32
lossi = []


for i in range(max_steps):
    # Forward pass
    ix = torch.randint(0, Xtr.shape[0], [batch_size,], generator = g)
    Xb, Yb = Xtr[ix], Ytr[ix]
    
    emb = C[Xb] # embedding
    embcat = emb.view(emb.shape[0],-1)
    hpreact = embcat @ w1 + b1
    # hpreact = (hpreact - hpreact.mean(0, keepdim = True)) / hpreact.std(0, keepdim=True) -> do this force all to be normalised
    # However, we need the normalisation to be more dynamic, it should unit gausian an standard, but can shift towars 1/-1 little
    # so we introduce bngain, and bnbias to output of hidden layer to allow slight elasticity
    bnmeani = hpreact.mean(0, keepdim = True)
    bnstdi = hpreact.std(0, keepdim=True)
    hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias
    h = torch.tanh(hpreact) #first layer output
    logits = h @ w2 + b2 # second layer output
    loss = F.cross_entropy(logits, Ytr[ix]) # use this rather than do the negative log likelihood repeat

    with torch.no_grad(): #ouside gradient optimisztation
        bnmean_running = 0.999 * bnmean_running + 0.001 * bnmeani # use those numbers because older means dont matter as much compared to more recent ones
        bnstd_running = 0.999 * bnstd_running + 0.001 * bnstd_running
    # backward
    for p in parameters:
        p.grad = None
    loss.backward()
    lr = 0.1 if i<100000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    if i% 10000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():4f}')
    lossi.append(loss.log10().item())

      0/ 200000: 3.658728
  10000/ 200000: 2.485005
  20000/ 200000: 2.381206
  30000/ 200000: 2.122497
  40000/ 200000: 1.976489
  50000/ 200000: 2.466371
  60000/ 200000: 2.293487
  70000/ 200000: 2.033890
  80000/ 200000: 1.979089
  90000/ 200000: 2.008217
 100000/ 200000: 2.442026
 110000/ 200000: 2.192243
 120000/ 200000: 2.159107
 130000/ 200000: 2.405686
 140000/ 200000: 2.273252
 150000/ 200000: 2.387841
 160000/ 200000: 2.147719
 170000/ 200000: 1.959215
 180000/ 200000: 2.319969
 190000/ 200000: 1.831223


In [48]:
@torch.no_grad()
def split_loss(split):
    x,y = {
        'train' : (Xtr, Ytr),
        'val' : (Xdev, Ydev),
        'test' : (Xte, Yte)
    }[split]
    emb = C[x]
    embcat = emb.view(emb.shape[0],-1)
    hpreact = embcat @ w1 + b1
    hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
    h = torch.tanh(hpreact) #first layer output
    
    logits = h @ w2 + b2 # second layer output
    loss = F.cross_entropy(logits, y) # use this rather than do the negative log likelihood repeat
    print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.3306777477264404
val 2.411884069442749


In [56]:
for _ in range (20):
    out = []
    context= [0] * block_size
    while True:
        emb = C[torch.tensor([context])] # first dim is 1 cause only 1 sample we gettting
        h = torch.tanh(emb.view(1,-1) @ w1 + b1) 
        logits = h @ w2 + b2 
        probs = F.softmax(logits, dim =1)
        # why?
        ix = torch.multinomial(probs, num_samples = 1, generator = g).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break
    print("".join(itos[i] for i in out))

strlnisssvannebriggevblanncadgenvatrichndvcallessphperdustransleylgqsondskkiordllissrl.
try.
qushngrnikblllsndhvannckshlnnbellancerplyn.
mckcrantan.
slidvtco.
swls.
kstlffrdidan.
swr.
srilliankarkshlssmukrysrivannstsvik.
kavp.
qumikkahmillswestlevodharlbid.
khnighuqjossylvaghb.
stemillqzavvontlffriffitrsubrttslfelbvo.
kstre.
kozhbdushmiarryazshefindbeltzslejquughbaqumummillarshadnlvontth.
trig.
qusrfichrrckspristycxajustydnm.
pryxe.
ststydy.
slonnverorfynn.


In [80]:
#Summary - similar to torch.Linear

class Linear:

    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn(fan_in,fan_out, generator = g) / fan_in**0.5 # kaimin
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x):
        self.out = x @self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])

class BatchNorm1d:

    def __init__(self,dim, eps = 1e-5, momentum = 0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self, x):
        if self.training:
            xmean = x.mean(0, keepdim = True)
            xvar = x.var(0, keepdim = True, unbiased = True)
        else:
            xmeean = self.running_mean
            xvar = self.running_var
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta
        if(self.training):
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xmean
        return self.out
    
    def parameters(self):
        return [self.gamma, self.beta]

class Tanh:
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out
    def parameters(self):
        return []


n_emb = 10 # number embedding dimenstions
n_hidden = 100 #number of neurons in hidden layer
g = torch.Generator().manual_seed(2147483647)

C = torch.randn((vocab_size, n_emb), generator = g)
layers = [
    Linear(n_emb * block_size, n_hidden), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, n_hidden), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, n_hidden), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, n_hidden), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, n_hidden), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, vocab_size), BatchNorm1d(vocab_size),
]
with torch.no_grad():
    # layers[-1].weight *= 0.1 # because we want the last layer initialised close to 0, since it doesnt have any acitvation and gives logits. to minimise initial loss
    layers[-1].gamma *= 0.1 # because of batch norm
    for layer in layers[:-1]:
        if isinstance(layer, Linear):
            layer.weight *= 5/3
parameters = [C] + [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

47551


In [81]:
# batch norm

max_steps = 200000
batch_size = 32
lossi = []


for i in range(max_steps):
    # Forward pass
    ix = torch.randint(0, Xtr.shape[0], [batch_size,], generator = g)
    Xb, Yb = Xtr[ix], Ytr[ix]
    
    emb = C[Xb] # embedding
    x = emb.view(emb.shape[0],-1)
    for layer in layers:
        x = layer(x)
    loss = F.cross_entropy(x, Yb)

    for layer in layers:
        layer.out.retain_grad() 
    for p in parameters:
        p.grad = None
    loss.backward()
    
    lr = 0.1 if i<100000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    if i% 10000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():4f}')
    lossi.append(loss.log10().item())
    break

      0/ 200000: 3.331266
